In [ ]:
#!pip install mamba-ssm --no-build-isolation -q
#!pip install transformers==4.45.0 -q

#import mamba_ssm
#print(mamba_ssm.__version__)

In [2]:
import torch
import torch.nn as nn
import numpy as np
from transformers import MambaModel, MambaConfig, AutoTokenizer
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    import causal_conv1d
    import mamba_ssm
    print("✅ CUDA Kernels loaded!")
except ImportError:
    print("❌ Slow mode")

os.environ["MAMBA_FORCE_BUILD"] = "FALSE"
device = "cuda" if torch.cuda.is_available() else "cpu"

def get_dct_matrix(n):
    dct = np.zeros((n, n))
    for k in range(n):
        for i in range(n):
            if k == 0:
                dct[k, i] = np.sqrt(1/n)
            else:
                dct[k, i] = np.sqrt(2/n) * np.cos(np.pi * k * (2*i + 1) / (2*n))
    return torch.tensor(dct, dtype=torch.float32)

class OrthogonalSubspace(nn.Module):
    def __init__(self, hidden_size, subspace_dim):
        super().__init__()
        self.subspace_dim = subspace_dim
        half = hidden_size // 2  # 384
        
        self.S_pos = nn.Linear(half, subspace_dim, bias=False)
        self.S_neg = nn.Linear(half, subspace_dim, bias=False)
        
        dct = get_dct_matrix(min(half, subspace_dim))
        nn.init.orthogonal_(self.S_pos.weight)
        nn.init.orthogonal_(self.S_neg.weight)
        
        self.S_pos.weight.requires_grad = False
        self.S_neg.weight.requires_grad = False
        
        self.half = half
    
    def forward(self, x):
        x_pos = x[:, :self.half]
        x_neg = x[:, self.half:]
        e_pos = torch.norm(self.S_pos(x_pos), dim=-1) / np.sqrt(self.subspace_dim)
        e_neg = torch.norm(self.S_neg(x_neg), dim=-1) / np.sqrt(self.subspace_dim)
        return e_pos, e_neg

class ContraMamba(nn.Module):
    def __init__(self, model_name, subspace_dim=256):
        super().__init__()
        config = MambaConfig.from_pretrained(model_name)
        config.use_mamba_kernels = True
        self.mamba = MambaModel.from_pretrained(model_name, config=config)
        hidden_size = self.mamba.config.hidden_size
        
        for name, param in self.mamba.named_parameters():
            if 'A_log' in name:
                param.requires_grad = False
        
        self.subspace = OrthogonalSubspace(hidden_size, subspace_dim)
        self.classifier = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    
    def forward(self, input_ids, labels=None):
        outputs = self.mamba(input_ids=input_ids)
        pooled = outputs.last_hidden_state.mean(dim=1)
        e_pos, e_neg = self.subspace(pooled)
        energy = torch.stack([e_pos, e_neg], dim=-1)
        logits = self.classifier(energy)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            base_loss = loss_fn(logits, labels)
            pos_mask = (labels == 2).float()
            neg_mask = (labels == 0).float()
            gap_loss = torch.mean(
                pos_mask * torch.relu(1.0 - (e_pos - e_neg)) +
                neg_mask * torch.relu(1.0 - (e_neg - e_pos))
            ) * 0.1
            loss = base_loss + gap_loss
        
        return {"loss": loss, "logits": logits, "e_pos": e_pos, "e_neg": e_neg}

model_v2 = ContraMamba("state-spaces/mamba-130m-hf").to(device)
print(f"✅ ContraMamba v2 로드 완료! 디바이스: {device.upper()}")
print(f"학습 파라미터: {sum(p.numel() for p in model_v2.parameters() if p.requires_grad):,}")

✅ CUDA Kernels loaded!
✅ ContraMamba v2 로드 완료! 디바이스: CUDA
학습 파라미터: 128,545,635


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
from datasets import load_dataset

# 1. 토크나이저와 데이터셋을 명시적으로 다시 정의
tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
dataset = load_dataset("boolq")

def preprocess(example):
    text = example['question'] + " " + example['passage']
    encoding = tokenizer(
        text,
        max_length=64,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    label = 2 if example['answer'] else 0
    return {
        'input_ids': encoding['input_ids'].squeeze(),
        'label': label
    }

train_data = dataset['train'].map(preprocess)
val_data = dataset['validation'].map(preprocess)
train_data.set_format(type='torch', columns=['input_ids', 'label'])
val_data.set_format(type='torch', columns=['input_ids', 'label'])

print("데이터 준비 완료")
print("train:", len(train_data), "val:", len(val_data))

데이터 준비 완료
train: 9427 val: 3270


In [4]:
import torch
import torch.nn as nn
import numpy as np
from transformers import MambaModel

def get_dct_matrix(n):
    """DCT 행렬 생성 - 열벡터들이 서로 직교"""
    dct = np.zeros((n, n))
    for k in range(n):
        for i in range(n):
            if k == 0:
                dct[k, i] = np.sqrt(1/n)
            else:
                dct[k, i] = np.sqrt(2/n) * np.cos(np.pi * k * (2*i + 1) / (2*n))
    return torch.tensor(dct, dtype=torch.float32)

class OrthogonalSubspace(nn.Module):
    def __init__(self, hidden_size, subspace_dim=256): # 차원은 256 추천
        super().__init__()
        self.subspace_dim = subspace_dim
        self.S_pos = nn.Linear(hidden_size, subspace_dim, bias=False)
        self.S_neg = nn.Linear(hidden_size, subspace_dim, bias=False)
        
        # 1. subspace_dim x (hidden_size // 2) 크기의 DCT 행렬을 만듭니다.
        # 384개 채널로부터 256개 특징을 뽑아내기 위함
        half_size = hidden_size // 2
        
        # 행렬 크기를 맞추기 위해 새로 DCT 생성
        def create_custom_dct(rows, cols):
            mat = np.zeros((rows, cols))
            for k in range(rows):
                for i in range(cols):
                    if k == 0:
                        mat[k, i] = np.sqrt(1/cols)
                    else:
                        mat[k, i] = np.sqrt(2/cols) * np.cos(np.pi * k * (2*i + 1) / (2*cols))
            return torch.tensor(mat, dtype=torch.float32)

        dct_half = create_custom_dct(subspace_dim, half_size)

        # 2. 물리적 격리: S_pos는 앞쪽만, S_neg는 뒤쪽만 값을 가짐
        with torch.no_grad():
            # S_pos: [256, 384] | [256, 384] 형태 중 앞쪽만 채움
            self.S_pos.weight.fill_(0)
            self.S_pos.weight[:, :half_size] = dct_half
            
            # S_neg: [256, 384] | [256, 384] 형태 중 뒤쪽만 채움
            self.S_neg.weight.fill_(0)
            self.S_neg.weight[:, half_size:] = dct_half

        # 3. 학습 허용 (이걸 풀어야 Mamba가 앞뒤 칸막이에 맞춰 정보를 배분함)
        self.S_pos.weight.requires_grad = True
        self.S_neg.weight.requires_grad = True

    def forward(self, x):
        # 차원 보정 (256차원 기준)
        e_pos = torch.norm(self.S_pos(x), dim=-1) / np.sqrt(self.subspace_dim)
        e_neg = torch.norm(self.S_neg(x), dim=-1) / np.sqrt(self.subspace_dim)
        return e_pos, e_neg

class ContraMamba(nn.Module):
    def __init__(self, model_name, subspace_dim=64, energy_threshold=0.5):
        super().__init__()
        self.mamba = MambaModel.from_pretrained(model_name)
        hidden_size = self.mamba.config.hidden_size
        self.energy_threshold = energy_threshold
        
        # A 행렬 freeze
        for name, param in self.mamba.named_parameters():
            if 'A_log' in name:
                param.requires_grad = False
        
        # 직교 subspace
        self.subspace = OrthogonalSubspace(hidden_size, subspace_dim)
        
        # 분류 헤드 (에너지 2개 → 3 클래스)
        self.classifier = nn.Sequential(
        nn.Linear(2, 16),
        nn.ReLU(),
        nn.Linear(16, 3)
        )
    
    def forward(self, input_ids, labels=None):
        outputs = self.mamba(input_ids=input_ids)
        pooled = outputs.last_hidden_state[:, -1, :]
        
        e_pos, e_neg = self.subspace(pooled)
        
        # 에너지 기반 분류
        energy = torch.stack([e_pos, e_neg], dim=-1)
        logits = torch.stack([e_neg, (e_pos + e_neg) / 2, e_pos], dim=-1)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        
        return {"loss": loss, "logits": logits, 
                "e_pos": e_pos, "e_neg": e_neg}
    
    def predict(self, input_ids):
        """에너지 threshold 기반 {-1, 0, 1} 출력"""
        output = self.forward(input_ids)
        e_pos = output['e_pos']
        e_neg = output['e_neg']
        
        pred = torch.zeros(input_ids.shape[0], dtype=torch.long)
        pred[e_pos > self.energy_threshold] = 2   # 1 (맞음)
        pred[e_neg > self.energy_threshold] = 0   # -1 (틀림)
        # 둘 다 threshold 이하 → 1 (모름)
        
        # 레이블 변환: 0→-1, 1→0, 2→1
        label_map = {0: -1, 1: 0, 2: 1}
        return pred, label_map

model_v2 = ContraMamba("state-spaces/mamba-130m-hf")
print("ContraMamba v2 로드 완료")
print(f"학습 파라미터 수: {sum(p.numel() for p in model_v2.parameters() if p.requires_grad):,}")
print(f"전체 파라미터 수: {sum(p.numel() for p in model_v2.parameters()):,}")

ContraMamba v2 로드 완료
학습 파라미터 수: 128,643,939
전체 파라미터 수: 129,233,763


In [5]:
# 에너지 분리 확인용
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_v2 = model_v2.to(device)

sample_input = torch.randint(0, 1000, (4, 32)).to(device)
with torch.no_grad():
    out = model_v2(sample_input)
    print("S+ 에너지:", out['e_pos'])
    print("S- 에너지:", out['e_neg'])
    print("같은 값?", torch.allclose(out['e_pos'], out['e_neg']))

# 훈련 방식
def train_v2_experiment(model, loader, optimizer):
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc="Ortho Training"):
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        
        # device_type 키워드 삭제 -> 'cuda'만 넣거나 아예 비워두기
        with autocast('cuda'): 
            output = model(input_ids=input_ids, labels=labels)
            
            base_loss = output['loss'].mean()
            e_pos, e_neg = output['e_pos'], output['e_neg']
            
            pos_mask = (labels == 2).float()
            neg_mask = (labels == 0).float()
            
            gap_loss = torch.mean(
                pos_mask * torch.relu(1.0 - (e_pos - e_neg)) + 
                neg_mask * torch.relu(1.0 - (e_neg - e_pos))
            )
            
            combined_loss = base_loss + 0.5 * gap_loss
            
        scaler.scale(combined_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += combined_loss.item()
        
    return total_loss / len(loader)

S+ 에너지: tensor([1.3762, 2.1654, 2.5328, 2.5338], device='cuda:0')
S- 에너지: tensor([0.9145, 2.4296, 2.5844, 2.6376], device='cuda:0')
같은 값? False


In [9]:
model_v2.load_state_dict(torch.load('/kaggle/working/contramamba_epoch2.pt'))
print("epoch 7 체크포인트 로드 완료")

epoch 7 체크포인트 로드 완료


In [13]:
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from torch.optim import AdamW
from tqdm.auto import tqdm
import gc

train_loader = DataLoader(train_data, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=4, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_v2 = model_v2.to(device)

optimizer_v2 = AdamW(model_v2.parameters(), lr=2e-5)
scaler = GradScaler('cuda')

def run_experiment(epochs=3, start_epoch=0):
    for epoch in range(epochs):
        # Training
        model_v2.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")
        
        for batch in pbar:
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            optimizer_v2.zero_grad()
            
            with autocast('cuda'):
                output = model_v2(input_ids=input_ids, labels=labels)
                base_loss = output['loss']
                e_pos, e_neg = output['e_pos'], output['e_neg']
                pos_mask = (labels == 2).float()
                neg_mask = (labels == 0).float()
                gap_loss = torch.mean(
                    pos_mask * torch.relu(1.0 - (e_pos - e_neg)) +
                    neg_mask * torch.relu(1.0 - (e_neg - e_pos))
                )
                combined_loss = base_loss + 0.1 * gap_loss

                if torch.isnan(combined_loss):
                    print("NaN detected, skipping batch")
                    optimizer_v2.zero_grad()
                    continue
            
                scaler.scale(combined_loss).backward()
                scaler.unscale_(optimizer_v2)
                torch.nn.utils.clip_grad_norm_(model_v2.parameters(), max_norm=1.0)
                scaler.step(optimizer_v2)
                scaler.update()
                
                pbar.set_postfix({'loss': f"{combined_loss.item():.4f}"})
            
        gc.collect()
        torch.cuda.empty_cache()   
        
        # Evaluation
        model_v2.eval()
        correct, total = 0, 0
        all_pos_e, all_neg_e = [], []
        pos_e_true, neg_e_true = [], []
        pos_e_false, neg_e_false = [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Eval]"):
                input_ids = batch['input_ids'].to(device)
                labels = batch['label'].to(device)
                
                output = model_v2(input_ids=input_ids)
                
                preds = output['logits'].argmax(dim=-1)
                correct += (preds == labels).sum().item()
                total += len(labels)
                
                e_pos = output['e_pos'].detach().cpu()
                e_neg = output['e_neg'].detach().cpu()
                labels_cpu = labels.cpu()
                
                all_pos_e.append(e_pos)
                all_neg_e.append(e_neg)
                
                # True 샘플(label==2)의 S+ 에너지
                true_mask = (labels_cpu == 2)
                false_mask = (labels_cpu == 0)
                
                if true_mask.any():
                    pos_e_true.append(e_pos[true_mask])
                    neg_e_true.append(e_neg[true_mask])
                if false_mask.any():
                    neg_e_false.append(e_neg[false_mask])
                    pos_e_false.append(e_pos[false_mask])
        
        mean_pos = torch.cat(all_pos_e).mean().item()
        mean_neg = torch.cat(all_neg_e).mean().item()
        mean_pos_true = torch.cat(pos_e_true).mean().item()
        mean_neg_true = torch.cat(neg_e_true).mean().item()
        mean_pos_false = torch.cat(pos_e_false).mean().item()
        mean_neg_false = torch.cat(neg_e_false).mean().item()
        
        all_pos_e.clear()
        all_neg_e.clear()
        pos_e_true.clear()
        neg_e_false.clear()
        
        gc.collect()
        torch.cuda.empty_cache()

        true_gap = mean_pos_true - mean_neg_true
        false_gap = mean_neg_false - mean_pos_false
        
        print(f"\n--- Epoch {epoch+1} Summary ---")
        print(f"Val Accuracy: {correct/total:.4f} | S+ Energy: {mean_pos:.4f} | S- Energy: {mean_neg:.4f}")
        print(f"True sample  → S+: {mean_pos_true:.4f} | S-: {mean_neg_true:.4f} | Gap: {true_gap:.4f}")
        print(f"False sample → S+: {mean_pos_false:.4f} | S-: {mean_neg_false:.4f} | Gap: {false_gap:.4f}")
        print(f"Energy Gap: {abs(mean_pos - mean_neg):.4f}")
        print("="*40)

        torch.save(model_v2.state_dict(), f'/kaggle/working/contramamba_epoch{epoch+1}.pt')
        print(f"체크포인트 저장 완료: contramamba_epoch{epoch+1}.pt")

run_experiment(epochs=3, start_epoch=7)

Epoch 1 [Train]:   0%|          | 0/1179 [00:00<?, ?it/s]

Epoch 1 [Eval]:   0%|          | 0/818 [00:00<?, ?it/s]


--- Epoch 1 Summary ---
Val Accuracy: 0.6618 | S+ Energy: 10.6567 | S- Energy: 5.8207
True샘플  → S+: 12.0468 | S-: 4.4647 | 갭: 7.5821
False샘플 → S+: 8.3720 | S-: 8.0494 | 갭: -0.3226
Energy Gap: 4.8360
체크포인트 저장 완료: contramamba_epoch1.pt


Epoch 2 [Train]:   0%|          | 0/1179 [00:00<?, ?it/s]

Epoch 2 [Eval]:   0%|          | 0/818 [00:00<?, ?it/s]


--- Epoch 2 Summary ---
Val Accuracy: 0.6581 | S+ Energy: 10.5905 | S- Energy: 6.6794
True샘플  → S+: 12.1026 | S-: 5.2199 | 갭: 6.8826
False샘플 → S+: 8.1055 | S-: 9.0779 | 갭: 0.9724
Energy Gap: 3.9112
체크포인트 저장 완료: contramamba_epoch2.pt


Epoch 3 [Train]:   0%|          | 0/1179 [00:00<?, ?it/s]

Epoch 3 [Eval]:   0%|          | 0/818 [00:00<?, ?it/s]


--- Epoch 3 Summary ---
Val Accuracy: 0.6584 | S+ Energy: 10.8609 | S- Energy: 6.4298
True샘플  → S+: 12.3219 | S-: 4.9801 | 갭: 7.3419
False샘플 → S+: 8.4597 | S-: 8.8124 | 갭: 0.3527
Energy Gap: 4.4311
체크포인트 저장 완료: contramamba_epoch3.pt


In [ ]:
# 체크포인트 저장 위치 확인
import os
print(os.listdir('/kaggle/working/'))